# 04 — ML Model: Weapon Crime Prediction
**Prerequisite:** `02_silver_lapd_crimes.ipynb` must have run.

**Goal:** Predict `Has_Weapon` (binary) using crime-base + NIBRS features.

**Approach:** Single Logistic Regression with fixed parameters — no CrossValidator.
- Logistic Regression is the simplest, fastest classifier for binary prediction
- No CV: avoids Databricks CE caching/OOM issues
- Fixed `regParam=0.01` (well-known good default for scaled features)

**Fix applied:** Uses Spark SQL `dense_rank()` for categorical encoding
instead of `StringIndexer` (blocked on Shared/Serverless clusters via Py4J whitelist).

## 1. Imports

In [0]:
import os; os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/default/raw_data/sparkml_temp"

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql.window import Window
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import gc, time
gc.collect()

SILVER_TBL = "silver_lapd_crimes"
MODEL_PATH = "/Volumes/workspace/default/raw_data/models/weapon_lr_model"
print("Imports OK")

Imports OK


## 2. Load Silver Table & Prepare Dataset

In [0]:
df = spark.table(SILVER_TBL)
print(f"Silver rows : {df.count():,}")

model_df = (
    df
    .filter(F.col("Has_Weapon").isNotNull())
    .filter(F.col("AREA").isNotNull())
    .filter(F.col("Hour").isNotNull())
    .filter(F.col("Month").isNotNull())
    .select(
        F.col("Has_Weapon").cast(IntegerType()).alias("label"),
        F.col("AREA").cast(DoubleType()),
        F.col("AREA_NAME"),
        F.col("Hour").cast(DoubleType()),
        F.col("Month").cast(DoubleType()),
        F.col("IsWeekend").cast(DoubleType()),
        F.col("Reporting_Delay").cast(DoubleType()),
        F.col("Vict_Age").cast(DoubleType()),
        F.col("Vict_Sex"),
        F.col("Vict_Descent"),
        F.col("Premise_Desc"),
        F.col("Crm_Cd_Desc"),
        F.col("Part_1_2").cast(DoubleType()),
        F.col("NIBR_Group"),
        F.col("Crime_Against"),
        F.col("DomesticViolence").cast(DoubleType()),
        F.col("HateCrime").cast(DoubleType()),
        F.col("GangRelated").cast(DoubleType()),
        F.col("HomelessVictim").cast(DoubleType()),
        F.col("Victim_Type"),
    )
    .fillna("Unknown", subset=["Vict_Sex", "Vict_Descent", "Premise_Desc",
                                "Crm_Cd_Desc", "NIBR_Group", "Crime_Against",
                                "Victim_Type", "AREA_NAME"])
    .fillna(0.0, subset=["Vict_Age", "IsWeekend", "Reporting_Delay", "Part_1_2",
                          "DomesticViolence", "HateCrime", "GangRelated",
                          "HomelessVictim", "AREA", "Hour", "Month"])
)

pos   = model_df.filter(F.col("label") == 1).count()
neg   = model_df.filter(F.col("label") == 0).count()
total = model_df.count()
print(f"\nModel dataset   : {total:,} rows")
print(f"  Weapon (1)    : {pos:,}  ({pos/total*100:.1f}%)")
print(f"  No Weapon (0) : {neg:,}  ({neg/total*100:.1f}%)")

Silver rows : 62,105

Model dataset   : 62,105 rows
  Weapon (1)    : 3,641  (5.9%)
  No Weapon (0) : 58,464  (94.1%)


## 3. Categorical Encoding via SQL (avoids StringIndexer whitelist issue)

Uses `dense_rank()` window function — same result as StringIndexer, works on all cluster types.

In [0]:
cat_cols = [
    "AREA_NAME", "Vict_Sex", "Vict_Descent",
    "Premise_Desc", "Crm_Cd_Desc",
    "NIBR_Group", "Crime_Against", "Victim_Type"
]

encoded_df = model_df
for col_name in cat_cols:
    freq_df = (
        model_df.groupBy(col_name)
        .count()
        .withColumn("_rank", F.dense_rank().over(
            Window.orderBy(F.desc("count"))
        ) - 1)
        .select(
            F.col(col_name).alias(f"_join_{col_name}"),
            F.col("_rank").cast(DoubleType()).alias(f"{col_name}_idx")
        )
    )
    encoded_df = (
        encoded_df
        .join(freq_df,
              encoded_df[col_name] == freq_df[f"_join_{col_name}"],
              "left")
        .drop(f"_join_{col_name}")
    )

idx_cols = [f"{c}_idx" for c in cat_cols]
encoded_df = encoded_df.fillna(0.0, subset=idx_cols)

print(f"Encoded {len(cat_cols)} categorical columns")
print(f"Encoded dataset: {encoded_df.count():,} rows, {len(encoded_df.columns)} cols")

Encoded 8 categorical columns


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Encoded dataset: 62,105 rows, 28 cols


## 4. Train / Test Split

In [0]:
train_df, test_df = encoded_df.randomSplit([0.8, 0.2], seed=42)
print(f"Train : {train_df.count():,}  |  Test : {test_df.count():,}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Train : 49,822  |  Test : 12,283


## 5. Feature Assembly

In [0]:
num_cols = [
    "AREA", "Hour", "Month", "IsWeekend",
    "Reporting_Delay", "Vict_Age", "Part_1_2",
    "DomesticViolence", "HateCrime", "GangRelated", "HomelessVictim"
]

feature_cols = idx_cols + num_cols

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="raw_features",
    handleInvalid="keep"
)

# StandardScaler important for Logistic Regression — normalises feature magnitudes
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withMean=True,
    withStd=True
)

print(f"Total features: {len(feature_cols)}  ({len(idx_cols)} encoded + {len(num_cols)} numeric)")

Total features: 19  (8 encoded + 11 numeric)


## 6. Logistic Regression (Fixed Parameters — No CV)

**Why Logistic Regression?**
- Simplest binary classifier — fast to train, low memory
- Works well on scaled tabular features
- Interpretable coefficients

**Fixed parameters (no CV):**
- `regParam=0.01` — light L2 regularisation, good default for scaled data
- `maxIter=100` — enough for convergence
- `elasticNetParam=0.0` — pure L2 (ridge)

In [0]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    regParam=0.01,
    maxIter=100,
    elasticNetParam=0.0,
    family="binomial"
)

lr_pipeline = Pipeline(stages=[assembler, scaler, lr])

evaluator_roc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)

print("Fitting Logistic Regression (regParam=0.01, maxIter=100)...")
t0 = time.time()
lr_model = lr_pipeline.fit(train_df)
elapsed = round(time.time() - t0, 2)
print(f"✓ Done in {elapsed}s")

lr_predictions = lr_model.transform(test_df)

lr_roc = evaluator_roc.evaluate(lr_predictions)
lr_acc = evaluator_acc.evaluate(lr_predictions)
lr_f1  = evaluator_f1.evaluate(lr_predictions)

print(f"\nLogistic Regression  ROC-AUC  : {lr_roc:.4f}")
print(f"Logistic Regression  Accuracy : {lr_acc:.4f}")
print(f"Logistic Regression  F1-Score : {lr_f1:.4f}")

Fitting Logistic Regression (regParam=0.01, maxIter=100)...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✓ Done in 49.69s

Logistic Regression  ROC-AUC  : 0.9123
Logistic Regression  Accuracy : 0.9385
Logistic Regression  F1-Score : 0.9205


## 7. Visualisations

In [0]:
# ── Metrics bar + Confusion Matrix ───────────────────────────────────────
cm_data = (
    lr_predictions
    .groupBy("label", "prediction")
    .count()
    .toPandas()
)

cm = np.zeros((2, 2))
for _, row in cm_data.iterrows():
    cm[int(row["label"])][int(row["prediction"])] = row["count"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Metrics bar
metrics_names = ["ROC-AUC", "Accuracy", "F1-Score"]
metrics_vals  = [lr_roc, lr_acc, lr_f1]
bar_colors    = ["steelblue", "darkorange", "mediumseagreen"]
bars = axes[0].bar(metrics_names, metrics_vals,
                   color=bar_colors, edgecolor="black", width=0.4)
for bar, val in zip(bars, metrics_vals):
    axes[0].text(bar.get_x() + bar.get_width() / 2, val + 0.005,
                 f"{val:.4f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
axes[0].set_ylim(0, 1.12)
axes[0].set_ylabel("Score")
axes[0].set_title("Logistic Regression — Evaluation Metrics")
axes[0].grid(True, alpha=0.3, axis="y")

# Confusion matrix
im = axes[1].imshow(cm, cmap="Blues")
plt.colorbar(im, ax=axes[1])
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(["Pred: No Weapon", "Pred: Weapon"])
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(["True: No Weapon", "True: Weapon"])
axes[1].set_title("Logistic Regression — Confusion Matrix")
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f"{int(cm[i,j]):,}",
                     ha="center", va="center", fontsize=13,
                     color="white" if cm[i, j] > cm.max() * 0.5 else "black")

plt.suptitle("Model Evaluation — Weapon Crime Prediction",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("/tmp/model_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# ── Top Feature Coefficients ─────────────────────────────────────────────
lr_stage = lr_model.stages[-1]
coeffs    = lr_stage.coefficients.toArray()
feat_names = [f"{c}_idx" for c in cat_cols] + num_cols

# Top 10 by absolute coefficient value
top10_idx = np.argsort(np.abs(coeffs))[-10:]
top10_names  = [feat_names[i] for i in top10_idx]
top10_coeffs = coeffs[top10_idx]

fig, ax = plt.subplots(figsize=(9, 5))
colors = ["coral" if v > 0 else "steelblue" for v in top10_coeffs]
ax.barh(top10_names, top10_coeffs, color=colors, edgecolor="black")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Coefficient value")
ax.set_title("Top 10 Feature Coefficients\n(coral = pushes toward Weapon, blue = away)")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig("/tmp/model_coefficients.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Save Model

In [0]:
lr_model.write().overwrite().save(MODEL_PATH)
print(f"✓ Logistic Regression model saved to: {MODEL_PATH}")
print(f"  regParam       : {lr_stage.getRegParam()}")
print(f"  maxIter        : {lr_stage.getMaxIter()}")
print(f"  elasticNetParam: {lr_stage.getElasticNetParam()}")

✓ Logistic Regression model saved to: /Volumes/workspace/default/raw_data/models/weapon_lr_model
  regParam       : 0.01
  maxIter        : 100
  elasticNetParam: 0.0


## 9. Results Summary

In [0]:
print("=" * 55)
print(f"{'Metric':<20} {'Score':>10}")
print("-" * 55)
print(f"{'ROC-AUC':<20} {lr_roc:>10.4f}")
print(f"{'Accuracy':<20} {lr_acc:>10.4f}")
print(f"{'F1-Score':<20} {lr_f1:>10.4f}")
print("=" * 55)

display(
    spark.createDataFrame(
        [("Logistic Regression", float(lr_roc), float(lr_acc), float(lr_f1))],
        ["Model", "ROC_AUC", "Accuracy", "F1_Score"]
    )
)

Metric                    Score
-------------------------------------------------------
ROC-AUC                  0.9123
Accuracy                 0.9385
F1-Score                 0.9205


Model,ROC_AUC,Accuracy,F1_Score
Logistic Regression,0.9123303621951391,0.9384515183587071,0.9204849616195574


---
## Summary

| Item | Detail |
|---|---|
| **Target** | `Has_Weapon` (binary: 0 = no weapon, 1 = weapon used) |
| **Model** | Logistic Regression (binomial) |
| **regParam** | 0.01 (L2 / ridge, no CV needed) |
| **maxIter** | 100 |
| **Encoding** | SQL `dense_rank()` (avoids StringIndexer Py4J whitelist issue) |
| **Scaler** | StandardScaler (mean=True, std=True) — required for LR |
| **Saved to** | `/Volumes/workspace/default/raw_data/models/weapon_lr_model` |